# EXP-03 XGBoost GPU · 재학습 (2023-06 시작)

팀 합의: Train 2023.06~2025.02, Dev 2025.03~04, Holdout 2025.05 미사용.

기존 EXP-03 모델 및 결과는 덮어쓰지 않습니다. Colab에서 T4 GPU를 선택하세요.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive")
projects=[ROOT/"2026 OIBC", ROOT/"2026_OIBC"]
relative_paths=["Data/data_v0.1/train_v0.1.parquet", "data/train_v0.1.parquet", "data_v0.1/train_v0.1.parquet"]
matches=[(p,p/rel) for p in projects for rel in relative_paths if (p/rel).is_file()]
if not matches: raise FileNotFoundError("Google Drive에서 train_v0.1.parquet를 찾지 못했습니다. 프로젝트 폴더 위치를 확인하세요.")
project,data_path=matches[0]
os.chdir(project)
os.environ["OIBC_TRAIN_PATH"]=str(data_path)
print("프로젝트:",project,"\n학습 데이터:",data_path)


In [ ]:
# Colab의 NumPy/pandas는 강제 업그레이드하지 않습니다.
# XGBoost가 없다는 오류가 나면 별도 셀에서 %pip install "xgboost>=2.1,<4" 실행 후 런타임을 다시 시작하세요.
import numpy as np
import pandas as pd
import xgboost as xgb
print("numpy", np.__version__, "pandas", pd.__version__, "xgboost", xgb.__version__)
!nvidia-smi


In [ ]:
"""
EXP-03: XGBoost CUDA GPU extended train from 2023-06 for OIBC 2026.

Run on Google Colab with a NVIDIA GPU (Runtime -> Change runtime type -> GPU):
    python exp03_xgb_train2306.py

Expected working directory:
    data/train_v0.1.parquet
    outputs/                  # optional: EXP-01 / EXP-02 error CSVs for comparison

Train/dev split and feature list match EXP-01.
Does NOT generate test predictions or an official VPP-group score.
"""

from __future__ import annotations

import gc
import os
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

EXP_ID = "exp03_xgb_train2306"
DATA_VERSION = "v0.1"
PROJECT_DIR = Path.cwd()
DATA_PATH = Path(os.environ.get("OIBC_TRAIN_PATH", str(PROJECT_DIR / "data" / "train_v0.1.parquet")))
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

# Colab: CUDA GPU. Local Mac CPU smoke-test only: change to "cpu".
DEVICE = "cuda"

TRAIN_START = pd.Timestamp("2023-06-01 01:00", tz="Asia/Seoul")
TRAIN_END = pd.Timestamp("2025-02-27 01:00", tz="Asia/Seoul")
DEV_START = pd.Timestamp("2025-03-01 01:00", tz="Asia/Seoul")
DEV_END = pd.Timestamp("2025-05-01 01:00", tz="Asia/Seoul")

CAT_FEATURES = ["site_id", "wa_station", "wb_station"]
NUM_FEATURES = [
    "capacity_kw", "lat", "lon", "wa_dist", "wb_dist",
    "hour_sin", "hour_cos", "doy_sin", "doy_cos",
    "elev", "azim", "cosz", "is_daylight",
    "wa_cloud_r10", "wa_cloud_r17",
    "wa_temp_r10", "wa_temp_r17",
    "wa_humidity_r10", "wa_humidity_r17",
    "wa_wind_speed_r10", "wa_wind_speed_r17",
    "wa_rain_r10", "wa_rain_r17",
    "wb_cloud_r10", "wb_cloud_r17",
    "wb_temp_r10", "wb_temp_r17",
    "wb_rel_hum_r10", "wb_rel_hum_r17",
    "wb_uv_idx_r10", "wb_uv_idx_r17",
    "wb_precip_prob_r10", "wb_precip_prob_r17",
    "wb_total_liq_r10", "wb_total_liq_r17",
    "wb_wind_speed_ms_r10", "wb_wind_speed_ms_r17",
    "fc_filled_a", "fc_filled_b", "fc_missing_a", "fc_missing_b",
]
FEATURES = CAT_FEATURES + NUM_FEATURES
LOAD_COLUMNS = list(dict.fromkeys(
    FEATURES + ["ts", "cf_target", "energy_kwh", "smp", "k"]
))

PARAMS = {
    "objective": "reg:absoluteerror",  # CF-target MAE; official metric is grouped penalty
    "eval_metric": "mae",
    "n_estimators": 800,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 15,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "max_bin": 128,
    "reg_lambda": 2.0,
    "tree_method": "hist",
    "device": DEVICE,
    "enable_categorical": True,
    "early_stopping_rounds": 60,
    "random_state": 42,
    "n_jobs": 4,
}


def require_cuda() -> None:
    if DEVICE != "cuda":
        print("[주의] DEVICE='cpu' — GPU 성능 실험이 아닙니다.")
        return
    try:
        result = subprocess.run(
            ["nvidia-smi"], capture_output=True, text=True, check=True
        )
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "NVIDIA GPU를 찾을 수 없습니다. Colab > 런타임 > 런타임 유형 변경 > GPU를 선택하세요. "
            "M3 Mac에서는 XGBoost CUDA GPU 학습이 불가능합니다."
        ) from exc
    print("NVIDIA GPU 확인:")
    print("\n".join(result.stdout.splitlines()[:10]))
    build_info = xgb.build_info()
    if str(build_info.get("USE_CUDA", "True")).lower() in {"false", "0", "off"}:
        raise RuntimeError("설치된 XGBoost에 CUDA 지원이 없습니다. Colab에서 xgboost를 다시 설치하세요.")


def load_data():
    if not DATA_PATH.is_file():
        raise FileNotFoundError(f"학습 데이터를 찾을 수 없습니다: {DATA_PATH}")

    print("\n[1/6] 데이터 읽기...")
    frame = pd.read_parquet(DATA_PATH, columns=LOAD_COLUMNS)
    print("원본 행 수:", f"{len(frame):,}")
    frame = frame.loc[
        (frame["ts"] >= TRAIN_START) & (frame["ts"] < DEV_END)
    ].copy()
    for col in CAT_FEATURES:
        frame[col] = frame[col].astype("category")
    for col in NUM_FEATURES:
        frame[col] = frame[col].astype("float32")

    train_mask = (frame["ts"] >= TRAIN_START) & (frame["ts"] < TRAIN_END)
    dev_mask = (frame["ts"] >= DEV_START) & (frame["ts"] < DEV_END)
    train = frame.loc[train_mask].copy()
    dev = frame.loc[dev_mask].copy()
    del frame
    gc.collect()

    if train.empty or dev.empty:
        raise ValueError("학습 또는 검증 데이터가 비어 있습니다.")
    if not (train["ts"].max() < dev["ts"].min()):
        raise ValueError("시간순 분할이 올바르지 않습니다.")
    if dev.duplicated(["site_id", "ts"]).any():
        raise ValueError("검증 데이터에 site_id-ts 중복이 있습니다.")
    if train["cf_target"].isna().any() or dev["cf_target"].isna().any():
        raise ValueError("정답에 결측이 있습니다.")
    if train["site_id"].nunique() != 253 or dev["site_id"].nunique() != 253:
        raise ValueError("학습 또는 검증 데이터의 발전소 수가 253개가 아닙니다.")

    print(f"학습: {len(train):,}행 / 검증: {len(dev):,}행 / 특징: {len(FEATURES)}개")
    print("Train:", train["ts"].min(), "~", train["ts"].max())
    print("Dev  :", dev["ts"].min(), "~", dev["ts"].max())
    return train, dev


def compare_saved_error_files(current_errors: pd.DataFrame) -> list[dict]:
    """Compare MAE/RMSE on EXACTLY identical (site_id, ts) and actuals."""
    print("\n[6/6] 저장된 EXP-01/02와 동일 행 비교...")
    ref = current_errors[["site_id", "ts", "actual_kwh"]].copy()
    ref["site_id"] = ref["site_id"].astype("string")
    ref["ts"] = pd.to_datetime(ref["ts"], utc=True)
    outcomes = []
    for exp in ("exp01_lgbm_train2306", "exp02_mlp_train2306"):
        path = OUTPUT_DIR / f"{exp}_dev_errors.csv"
        if not path.exists():
            print(f"  - {exp}: 파일 없음 (비교 생략)")
            continue
        try:
            other = pd.read_csv(path, usecols=["site_id", "ts", "actual_kwh", "pred_kwh"])
            other["site_id"] = other["site_id"].astype("string")
            other["ts"] = pd.to_datetime(other["ts"], utc=True)
            if len(other) != len(ref) or other.duplicated(["site_id", "ts"]).any():
                raise ValueError("행 수 불일치 또는 중복")
            paired = ref.merge(
                other, on=["site_id", "ts"], how="inner", validate="one_to_one", suffixes=("_ref", "_other")
            )
            if len(paired) != len(ref):
                raise ValueError("발전소·시각 조합 불일치")
            if not np.allclose(
                paired["actual_kwh_ref"].to_numpy(dtype=np.float64),
                paired["actual_kwh_other"].to_numpy(dtype=np.float64),
                rtol=1e-6, atol=1e-4,
            ):
                raise ValueError("실제 발전량 값 불일치")
            difference = (
                paired["pred_kwh"].to_numpy(dtype=np.float64)
                - paired["actual_kwh_ref"].to_numpy(dtype=np.float64)
            )
            outcomes.append({
                "experiment": exp,
                "rows": int(len(paired)),
                "mae_kwh": float(np.mean(np.abs(difference))),
                "rmse_kwh": float(np.sqrt(np.mean(difference ** 2))),
            })
        except (ValueError, KeyError, OSError) as exc:
            print(f"  - {exp}: 안전한 동일조건 비교 불가 ({exc})")

    return outcomes


def main():
    print("=" * 62)
    print(f"{EXP_ID} / data={DATA_VERSION} / XGBoost {xgb.__version__} / device={DEVICE}")
    print("=" * 62)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    require_cuda()
    train, dev = load_data()
    X_train = train[FEATURES]
    y_train = train["cf_target"].to_numpy(dtype=np.float32, copy=True)
    X_dev = dev[FEATURES]
    y_dev = dev["cf_target"].to_numpy(dtype=np.float32, copy=True)

    print("\n[2/6] XGBoost 학습 시작...")
    t0 = time.perf_counter()
    model = xgb.XGBRegressor(**PARAMS)
    model.fit(X_train, y_train, eval_set=[(X_dev, y_dev)], verbose=50)
    elapsed = time.perf_counter() - t0
    trained_device = json.loads(model.get_booster().save_config())["learner"]["generic_param"]["device"]
    print(f"사용 장치: {trained_device} / 최적 iteration: {model.best_iteration}")
    if DEVICE == "cuda" and not str(trained_device).startswith("cuda"):
        raise RuntimeError("GPU를 요청했지만 CPU로 실행되었습니다. CUDA 환경을 확인하세요.")

    print("\n[3/6] 발전량 예측 및 평가...")
    cf_pred = model.predict(X_dev)
    pred_kwh = np.maximum(cf_pred.astype(np.float64), 0.0) * dev["capacity_kw"].to_numpy(dtype=np.float64)
    pred_kwh = np.where(dev["is_daylight"].to_numpy() == 0, 0.0, pred_kwh)
    actual_kwh = dev["energy_kwh"].to_numpy(dtype=np.float64)
    if not np.isfinite(pred_kwh).all():
        raise ValueError("비정상 예측값(NaN/Inf)이 존재합니다.")
    err = pred_kwh - actual_kwh
    abserr = np.abs(err)
    day_mask = dev["is_daylight"].to_numpy() == 1
    day_mae = float(np.mean(abserr[day_mask])) if day_mask.any() else None
    weights = dev["smp"].to_numpy(dtype=np.float64) * dev["k"].to_numpy(dtype=np.float64)
    proxy = float(0.1 * np.sum(weights * abserr)) if np.isfinite(weights).all() else None
    metrics = {
        "experiment": EXP_ID, "data_version": DATA_VERSION,
        "device": trained_device, "xgboost_version": xgb.__version__,
        "train_start": str(TRAIN_START), "train_end_exclusive": str(TRAIN_END),
        "dev_start": str(DEV_START), "dev_end_exclusive": str(DEV_END),
        "train_rows": int(len(train)), "dev_rows": int(len(dev)),
        "features": FEATURES, "parameters": PARAMS,
        "best_iteration": int(model.best_iteration), "training_seconds": round(elapsed, 2),
        "mae_kwh": float(np.mean(abserr)),
        "rmse_kwh": float(np.sqrt(np.mean(err ** 2))),
        "daylight_mae_kwh": day_mae,
        "site_weighted_penalty_proxy": proxy,
        "warning": "site-weighted penalty proxy is NOT the official VPP group penalty",
    }
    print(f"MAE={metrics['mae_kwh']:.4f} kWh")
    print(f"RMSE={metrics['rmse_kwh']:.4f} kWh")
    print(f"주간 MAE={day_mae:.4f} kWh" if day_mae is not None else "주간 없음")
    print(f"발전소별 가중오차 참고값={proxy:,.2f}" if proxy is not None else "가중치 결측: 참고값 계산 생략")

    print("\n[4/6] 모델과 결과 파일 저장...")
    model_file = MODEL_DIR / f"{EXP_ID}.json"
    pred_file = OUTPUT_DIR / f"{EXP_ID}_dev_predictions.csv"
    error_file = OUTPUT_DIR / f"{EXP_ID}_dev_errors.csv"
    metrics_file = OUTPUT_DIR / f"{EXP_ID}_metrics.json"
    imp_file = OUTPUT_DIR / f"{EXP_ID}_feature_importance.csv"
    model.save_model(str(model_file))  # JSON retains categorical split information

    errors = dev[["site_id", "ts"]].copy()
    errors["actual_kwh"] = actual_kwh
    errors["pred_kwh"] = pred_kwh
    errors["error_kwh"] = err
    errors = errors.sort_values(["site_id", "ts"])
    if errors.duplicated(["site_id", "ts"]).any():
        raise ValueError("저장 전 중복 키 발견")
    errors[["site_id", "ts", "pred_kwh"]].to_csv(pred_file, index=False)
    errors[["site_id", "ts", "actual_kwh", "pred_kwh", "error_kwh"]].to_csv(error_file, index=False)
    pd.DataFrame({"feature": FEATURES, "importance": model.feature_importances_}).sort_values(
        "importance", ascending=False
    ).to_csv(imp_file, index=False)

    print("\n[5/6] 실험 메타데이터 기록...")
    comparisons = compare_saved_error_files(errors)
    comparisons.append({
        "experiment": EXP_ID, "rows": int(len(errors)),
        "mae_kwh": metrics["mae_kwh"], "rmse_kwh": metrics["rmse_kwh"],
    })
    metrics["comparison"] = comparisons
    with metrics_file.open("w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    pd.DataFrame(comparisons).sort_values("mae_kwh").to_csv(
        OUTPUT_DIR / "exp01_exp02_exp03_train2306_mae_comparison.csv", index=False
    )

    print("\n===== 동일 구간 모델 비교 (발전소별 지표) =====")
    print(pd.DataFrame(comparisons).sort_values("mae_kwh").to_string(index=False))
    print("\n저장 완료:")
    for p in (model_file, pred_file, error_file, imp_file, metrics_file):
        print(" -", p)
    print("\n참고: 공식 대회 점수는 두 그룹별 가중 절대오차를 합산해야 합니다.")


if __name__ == "__main__":
    main()
